## Scryfall Artwork Clustering Model

In [1]:
import altair as alt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ast import literal_eval

from sklearn.preprocessing import MultiLabelBinarizer
from sklearn import manifold
from sklearn.metrics import pairwise_distances

In [2]:
df = pd.read_csv('./data/final-scryfall-unique-artwork.csv')
creature_df = df[df['type'].str.contains('Creature')]
creature_df.head()

,id,oracle_id,name,set_id,set,set_name,artist_ids,artist,released_at,type_line,...,power,toughness,edhrec_rank,type,subtype,legality_commander,legality_standard,price_usd,image_uri_normal,image_uri_art_crop
1,0000579f-7b35-4ed3-b44c-db2a538066fe,44623693-51d6-49ad-8cd7-140505caf02f,Fury Sliver,c1d109bc-ffd8-428f-8d7d-3f8d7e648046,tsp,Time Spiral,['d48dd097-720d-476a-8722-6a02854ae28b'],Paolo Parente,2006-10-06,Creature — Sliver,...,3,3,9808.0,Creature,Sliver,legal,not_legal,0.46,https://cards.scryfall.io/normal/front/0/0/000...,https://cards.scryfall.io/art_crop/front/0/0/0...
2,00006596-1166-4a79-8443-ca9f82e6db4e,8ae3562f-28b7-4462-96ed-be0cf7052ccc,Kor Outfitter,eb16a2bd-a218-4e4e-8339-4aa1afc0c8d2,zen,Zendikar,['aa7e89ed-d294-4633-9057-ce04dacfcfa4'],Kieran Yanner,2009-10-02,Creature — Kor Soldier,...,2,2,19672.0,Creature,Kor Soldier,legal,not_legal,0.11,https://cards.scryfall.io/normal/front/0/0/000...,https://cards.scryfall.io/art_crop/front/0/0/0...
3,0000cd57-91fe-411f-b798-646e965eec37,9f0d82ae-38bf-45d8-8cda-982b6ead1d72,Siren Lookout,fe0dad85-54bc-4151-9200-d68da84dd0f2,xln,Ixalan,['a8e7b854-b15a-421a-b66d-6e68187ae285'],Chris Rallis,2017-09-29,Creature — Siren Pirate,...,1,2,18843.0,Creature,Siren Pirate,legal,not_legal,0.04,https://cards.scryfall.io/normal/front/0/0/000...,https://cards.scryfall.io/art_crop/front/0/0/0...
4,0001f1ef-b957-4a55-b47f-14839cdbab6f,ef027846-be81-4959-a6b5-56bd01b1e68a,Venerable Knight,a90a7b2f-9dd8-4fc7-9f7d-8ea2797ec782,eld,Throne of Eldraine,['9c201dbe-db56-429a-87e6-189ea70c2632'],Colin Boyer,2019-10-04,Creature — Human Knight,...,2,1,18404.0,Creature,Human Knight,legal,not_legal,0.15,https://cards.scryfall.io/normal/front/0/0/000...,https://cards.scryfall.io/art_crop/front/0/0/0...
6,0002ab72-834b-4c81-82b1-0d2760ea96b0,645b5784-a6f7-4cf3-966a-e1a51420b96b,Mystic Skyfish,bc94aba1-7376-4e02-a12d-3a2efb66ab0f,m21,Core Set 2021,['bb677b1a-ce51-4888-83d6-5a94de461ff9'],Alayna Danner,2020-07-03,Creature — Fish,...,3,1,23732.0,Creature,Fish,legal,not_legal,0.10,https://cards.scryfall.io/normal/front/0/0/000...,https://cards.scryfall.io/art_crop/front/0/0/0...


In [3]:
print(len(creature_df))

20408


In [4]:
# calculate the 99th percentile threshold, then filter to get top 1% prices
creature_df.loc[:,'price_usd'] = creature_df['price_usd'].round(2)
threshold = creature_df['price_usd'].quantile(0.99)
top_percent_df = creature_df[creature_df['price_usd'] >= threshold].sort_values(by='price_usd').reset_index()
print(len(top_percent_df))

price_min = top_percent_df['price_usd'].min()
price_max = top_percent_df['price_usd'].max()
print(f"The price range is: {price_min} - {price_max}")


205
The price range is: 48.69 - 7618.21


### One-Hot Encoding

Using `MultiLabelBinarizer` since the cards can have multiple keywords (e.g. ['Flying', 'Trample']) and color identities (e.g. ['B', 'W']). The same applies for the subtype - if the subtype is 'Human Knight', we will create the encoding as Human-1, Knight-1. This way, human creatures and knight creatures will cluster, rather than the distinct human knights.

In [5]:
one_hot_encoding_df = top_percent_df.copy()
one_hot_encoding_df['subtype'] = top_percent_df['subtype'].fillna('').str.split()

In [6]:
mlb = MultiLabelBinarizer()
encoded_subtypes = mlb.fit_transform(one_hot_encoding_df['subtype'])
subtypes_df = pd.DataFrame(encoded_subtypes, columns='s_'+mlb.classes_, index=one_hot_encoding_df.index)
subtypes_df.head()

,s_Advisor,s_Angel,s_Archer,s_Artificer,s_Assassin,s_Avatar,s_Badger,s_Barbarian,s_Bard,s_Basilisk,...,s_Troll,s_Vampire,s_Wall,s_Warlock,s_Warrior,s_Wizard,s_Wolf,s_Wraith,s_Wurm,s_Zombie
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0


In [7]:
encoded_keywords = mlb.fit_transform(one_hot_encoding_df['keywords'].apply(literal_eval))
keywords_df = pd.DataFrame(encoded_keywords, columns='k_'+mlb.classes_, index=one_hot_encoding_df.index)
keywords_df.head()

,k_Amass,k_Ascend,k_Banding,k_Cascade,k_Deathtouch,k_Defender,k_Earthbend,k_Evoke,k_First strike,k_Flash,...,k_Rampage,k_Reach,k_Scry,k_Shadow,k_Swampwalk,k_Trample,k_Treasure,k_Vigilance,k_Ward,k_Warp
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [8]:
encoded_colors = mlb.fit_transform(one_hot_encoding_df['color_identity'].apply(literal_eval))
colors_df = pd.DataFrame(encoded_colors, columns='c_'+mlb.classes_, index=one_hot_encoding_df.index)
colors_df.head()

,c_B,c_G,c_R,c_U,c_W
0,0,1,0,0,0
1,0,1,1,1,1
2,1,0,0,0,0
3,0,0,0,0,0
4,0,0,1,0,0


In [9]:
features = pd.concat([subtypes_df, keywords_df, colors_df], axis=1)
features.head()

,s_Advisor,s_Angel,s_Archer,s_Artificer,s_Assassin,s_Avatar,s_Badger,s_Barbarian,s_Bard,s_Basilisk,...,k_Trample,k_Treasure,k_Vigilance,k_Ward,k_Warp,c_B,c_G,c_R,c_U,c_W
0,0,0,0,0,0,0,0,0,0,0,...,1,0,0,0,0,0,1,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,1,1,1,1
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


### MDS

In [10]:
# calculate dissimilarity (distance) matrix since data is binary/categorical
distance_matrix = pairwise_distances(features.to_numpy(), metric='jaccard') 

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/pairwise.py:2459: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


In [11]:
seed = np.random.RandomState(seed=42)
mds = manifold.MDS(n_components=2, metric='precomputed', max_iter=3000, eps=1e-9, random_state=seed, n_jobs=1, normalized_stress=True, n_init=1, init='classical_mds')
pos = mds.fit(distance_matrix).embedding_
top_percent_df['x'] = [x[0] for x in pos]
top_percent_df['y'] = [x[1] for x in pos]

stress = mds.stress_
print(f"MDS stress value: {stress}")

MDS stress value: 0.3783848990195919


In [12]:
def genMDSPlot(df, top_n=None, art_crop=True):
    # input: df -- dataframe (augmented with the x/y columns)
    # input: top_n -- top n of cards to display by price, if None show all cards (top 1%)
    # input: art_crop -- True to use cropped art image, False to use full card image
    # return: an altair chart (e.g., return alt.Chart(...))

    if top_n is None:
        top_n = len(df)
    top_n_cards = df.nlargest(n=top_n, columns='price_usd').sort_index()

    image_url = 'image_uri_art_crop' if art_crop else 'image_uri_normal'
    
    images = alt.Chart(top_n_cards).mark_image(width=40, height=40).encode(
        x=alt.X('x', axis=None),
        y=alt.Y('y', axis=None),
        url=image_url,
        tooltip=[
            alt.Tooltip("name"),
            alt.Tooltip("subtype"),
            alt.Tooltip("keywords"),
            alt.Tooltip("color_identity"),
            alt.Tooltip("price_usd")
        ]
    ).properties(
        width=1000,
        height=1000
    )

    squares = alt.Chart(top_n_cards).mark_square(size=2000).encode(
        x=alt.X('x', axis=None),
        y=alt.Y('y', axis=None),
        color=alt.Color('price_usd:Q', scale=alt.Scale(scheme='viridis'), legend=alt.Legend(title="Price (USD)"))
    )

    return squares + images

In [13]:
chart = genMDSPlot(top_percent_df)
chart


alt.LayerChart(...)

In [14]:
# save interactive chart as HTML
chart.save('artwork-cluster-vis.html')

In [15]:
# show only top 50 cards with full card image
genMDSPlot(top_percent_df, 50, art_crop=False)

alt.LayerChart(...)